# 🎯 Will they get placed? Classification end-to-end
Imbalance, leakage, tuning, calibration and a fairness check on the Indian CS Student Survival dataset.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
D = pd.read_csv('/kaggle/input/datasets/pranavsinghdeveloper/desi-cs-student-survival/students.csv')
S = pd.read_csv('/kaggle/input/datasets/pranavsinghdeveloper/desi-cs-student-survival/semester_records.csv')
F = D[D.year_of_study==4].copy(); F['y']=(F.placed=='Yes').astype(int); print(F.shape, F.y.mean().round(3))

## Leakage check
`package_lpa`, `offers_received` and `placement_category` are consequences of the outcome — they must never be features.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer; from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler; from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression; from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
drop=['placed','placement_category','offers_received','package_lpa','mess_review_text','reason_for_last_bunk','student_id','archetype','y']
X=F.drop(columns=drop); y=F.y
cat=X.select_dtypes(include=['object','bool']).columns.tolist(); num=[c for c in X.columns if c not in cat]
pre=ColumnTransformer([('n',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('c',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat)])
models={'LogReg':LogisticRegression(max_iter=2000),'RandForest':RandomForestClassifier(300,random_state=0,n_jobs=-1),'HistGB':HistGradientBoostingClassifier(random_state=0)}
for n,m in models.items(): print(n, cross_val_score(make_pipeline(pre,m),X,y,cv=5,scoring='roc_auc').mean().round(3))

## What matters? Permutation importance

In [ ]:
from sklearn.inspection import permutation_importance
Xtr,Xte,ytr,yte=train_test_split(X,y,random_state=0,stratify=y)
pipe=make_pipeline(pre,LogisticRegression(max_iter=2000)).fit(Xtr,ytr)
r=permutation_importance(pipe,Xte,yte,scoring='roc_auc',n_repeats=5,random_state=0)
imp=pd.Series(r.importances_mean,index=X.columns).sort_values().tail(12); imp.plot.barh(figsize=(7,4),title='Permutation importance (ROC-AUC drop)'); plt.show()

## Fairness audit
`gender` has **no causal effect** in the generator. A healthy model should barely use it.

In [ ]:
print('Importance of gender:', round(pd.Series(r.importances_mean,index=X.columns)['gender'],4))
te=Xte.assign(p=pipe.predict_proba(Xte)[:,1],y=yte); print(te.groupby('gender').agg(mean_pred=('p','mean'),actual=('y','mean'),n=('y','size')).round(3))

## Takeaways
* CGPA, LeetCode, internships, backlogs and college tier drive placement.
* Always drop post-outcome columns.
* Can you beat the baseline AUC in `challenges/CHALLENGES.md`?

---
Dataset docs, generator & challenges: https://github.com/Pranav-Singh-Devloper/indian-cs-student-survival-dataset